# Cohort Retention Analysis: Digital Subscription Service
**Analyst:** Zari Syed | **Date:** May 2025 | **Domain:** Subscription retention & customer lifetime value

---

## Overview

Cohort analysis tracks groups of subscribers who joined in the same period, revealing how long they stay,
when they churn, and what their lifetime value is. This is a foundational technique for any subscription business.

**Dataset:** 12 monthly subscriber cohorts (Jan-Dec 2024), ~480-650 subscribers per cohort

**Key questions:**
1. How does retention vary across cohorts?
2. At what point do we lose most subscribers?
3. What is the estimated customer lifetime value (LTV)?
4. Are newer cohorts performing better or worse than older ones?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titlesize': 13,
    'axes.labelsize': 11
})
np.random.seed(0)
print("Libraries loaded.")

## 1. Data Generation

We simulate a realistic subscription dataset with 12 monthly cohorts.

Key assumptions:
- Month-1 retention ~72% (high early churn is typical for media subscriptions)
- Retention stabilises around month 6 (~45-50% remaining)
- Newer cohorts have slightly better retention, reflecting product improvements

In [ ]:
MONTHS = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
N_COHORTS = 12
MAX_AGE = 12

base_retention = np.array([
    1.000, 0.720, 0.610, 0.555, 0.515, 0.485,
    0.463, 0.448, 0.436, 0.427, 0.420, 0.415
])
cohort_sizes = np.array([480, 495, 510, 530, 545, 570, 580, 600, 615, 625, 640, 650])
improvement = np.linspace(0, 0.04, N_COHORTS)

retention_matrix = np.zeros((N_COHORTS, MAX_AGE))
for i in range(N_COHORTS):
    curve = base_retention + improvement[i] * (1 - base_retention)
    available = MAX_AGE - i
    for m in range(min(available, MAX_AGE)):
        noise = np.random.normal(0, 0.008)
        rate = np.clip(curve[m] + noise, 0, 1)
        if m > 0:
            rate = min(rate, retention_matrix[i, m-1])
        retention_matrix[i, m] = rate
    for m in range(available, MAX_AGE):
        retention_matrix[i, m] = np.nan

print("Cohort acquisition sizes:")
for month, n in zip(MONTHS, cohort_sizes):
    print("  {} 2024: {:,}".format(month, n))
print("Total acquired in 2024: {:,}".format(cohort_sizes.sum()))

## 2. Retention Heatmap

The retention matrix shows what percentage of each original cohort is still subscribed at each month.
Reading across a row shows how a single cohort ages; reading down a column compares cohorts at the same lifecycle stage.

In [ ]:
ret_df = pd.DataFrame(
    retention_matrix * 100,
    index=['{} 2024'.format(m) for m in MONTHS],
    columns=['Month {}'.format(i) for i in range(MAX_AGE)]
)
mask = np.isnan(retention_matrix * 100)

fig, ax = plt.subplots(figsize=(14, 7))
sns.heatmap(ret_df, annot=True, fmt='.0f', cmap='YlGnBu',
            linewidths=0.5, linecolor='white',
            annot_kws={'size': 9}, mask=mask,
            cbar_kws={'label': 'Retention Rate (%)', 'shrink': 0.8}, ax=ax)
ax.set_title('Subscriber Retention by Cohort (% of original cohort still active)', pad=15)
ax.set_xlabel('Months Since Acquisition')
ax.set_ylabel('Acquisition Cohort')
plt.tight_layout()
plt.show()

## 3. Retention Curves by Cohort

Plotting each cohort's retention curve over time shows whether newer cohorts are performing differently.
Green = later cohorts, Red = earlier cohorts.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
cmap = plt.cm.RdYlGn(np.linspace(0.2, 0.85, N_COHORTS))

for i in range(N_COHORTS):
    available = N_COHORTS - i
    months = list(range(available))
    rates = retention_matrix[i, :available] * 100
    ax.plot(months, rates, '-o', color=cmap[i], markersize=4,
            linewidth=1.8, label='{} 2024'.format(MONTHS[i]), alpha=0.85)

ax.set_xlabel('Months Since Acquisition')
ax.set_ylabel('Retention Rate (%)')
ax.set_title('Subscriber Retention Curves by Acquisition Cohort')
ax.axhline(50, color='red', ls='--', lw=1, alpha=0.4, label='50% retention line')
ax.set_ylim(0, 110)
ax.legend(loc='upper right', fontsize=8, ncol=2, title='Cohort')
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
plt.tight_layout()
plt.show()

## 4. Average Retention & Churn Rate

Averaging across all cohorts reveals the typical subscriber lifecycle.
The churn rate chart identifies which months see the steepest drop-offs — the highest-priority intervention points.

In [ ]:
avg_retention = np.nanmean(retention_matrix, axis=0) * 100

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.plot(range(MAX_AGE), avg_retention, 'o-', color='#2C5F8A', lw=2.5, ms=7)
ax.fill_between(range(MAX_AGE), avg_retention, alpha=0.15, color='#2C5F8A')
ax.set_xlabel('Months Since Acquisition')
ax.set_ylabel('Average Retention Rate (%)')
ax.set_title('Average Retention Curve Across All Cohorts')
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
for m, r in enumerate(avg_retention):
    if m % 2 == 0:
        ax.annotate('{:.0f}%'.format(r), (m, r),
                    textcoords='offset points', xytext=(0, 8), ha='center', fontsize=8)

ax2 = axes[1]
churn_rate = np.diff(avg_retention) * -1
ax2.bar(range(1, MAX_AGE), churn_rate, color='#C0504D', alpha=0.75)
ax2.set_xlabel('Month of Subscription')
ax2.set_ylabel('Churn Rate (pp drop in retention)')
ax2.set_title('Average Monthly Churn Rate')
ax2.set_xticks(range(1, MAX_AGE))

plt.suptitle('Subscriber Lifecycle Analysis', fontsize=14)
plt.tight_layout()
plt.show()

print("Key retention milestones (average across cohorts):")
for m in [1, 2, 3, 6, 11]:
    print("  Month {:2d}: {:.1f}% still subscribed".format(m, avg_retention[m]))

## 5. Customer Lifetime Value (LTV)

LTV = expected lifetime (months) x monthly revenue x gross margin.
The expected lifetime is calculated as the area under each cohort's retention curve.

In [ ]:
MONTHLY_PRICE = 9.99
GROSS_MARGIN  = 0.70

ltv_data = []
for i in range(N_COHORTS):
    valid = retention_matrix[i][~np.isnan(retention_matrix[i])]
    lifetime_months = float(valid.sum())
    ltv = lifetime_months * MONTHLY_PRICE * GROSS_MARGIN
    ltv_data.append({
        'Cohort': '{} 2024'.format(MONTHS[i]),
        'Cohort_Size': int(cohort_sizes[i]),
        'Expected_Lifetime_Months': round(lifetime_months, 2),
        'LTV_GBP': round(ltv, 2)
    })

ltv_df = pd.DataFrame(ltv_data)
print(ltv_df.to_string(index=False))
print("\nAverage LTV: GBP{:.2f}  |  Avg expected lifetime: {:.1f} months".format(
    ltv_df['LTV_GBP'].mean(), ltv_df['Expected_Lifetime_Months'].mean()))

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar(ltv_df['Cohort'], ltv_df['LTV_GBP'], color='#4BACC6', alpha=0.85)
ax.axhline(ltv_df['LTV_GBP'].mean(), color='#E36C09', ls='--', lw=1.8,
           label='Avg LTV: GBP{:.2f}'.format(ltv_df['LTV_GBP'].mean()))
for bar, val in zip(bars, ltv_df['LTV_GBP']):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
            'GBP{:.0f}'.format(val), ha='center', va='bottom', fontsize=8.5)
ax.set_xlabel('Acquisition Cohort')
ax.set_ylabel('Estimated LTV (GBP)')
ax.set_title('Customer Lifetime Value by Acquisition Cohort')
ax.legend()
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

## 6. Cohort Trend Analysis

Tracking Month-3 and Month-6 retention across cohorts shows whether retention is improving over time,
indicating whether product or onboarding changes are having a positive effect.

In [ ]:
m3 = [retention_matrix[i, 3]*100 for i in range(N_COHORTS) if not np.isnan(retention_matrix[i, 3])]
m6 = [retention_matrix[i, 6]*100 for i in range(N_COHORTS) if not np.isnan(retention_matrix[i, 6])]
cohorts_m3 = MONTHS[:len(m3)]
cohorts_m6 = MONTHS[:len(m6)]

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(cohorts_m3, m3, 'o-', color='#2C5F8A', lw=2, ms=7, label='Month-3 retention')
ax.plot(cohorts_m6, m6, 's--', color='#70AD47', lw=2, ms=7, label='Month-6 retention')
if len(m3) > 2:
    z3 = np.polyfit(range(len(m3)), m3, 1)
    ax.plot(cohorts_m3, np.poly1d(z3)(range(len(m3))), ':', color='#2C5F8A', alpha=0.5)
if len(m6) > 2:
    z6 = np.polyfit(range(len(m6)), m6, 1)
    ax.plot(cohorts_m6, np.poly1d(z6)(range(len(m6))), ':', color='#70AD47', alpha=0.5)
ax.set_xlabel('Acquisition Cohort')
ax.set_ylabel('Retention Rate (%)')
ax.set_title('Month-3 and Month-6 Retention Trends (dotted = trend line)')
ax.legend()
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
plt.tight_layout()
plt.show()

## 7. Key Findings & Recommendations

In [ ]:
avg_ret = np.nanmean(retention_matrix, axis=0) * 100

print("=" * 60)
print("KEY FINDINGS")
print("=" * 60)
print("")
print("1. CHURN PATTERN")
print("   - Biggest drop: Month 0->1 ({:.0f}% -> {:.0f}%)".format(avg_ret[0], avg_ret[1]))
print("   - Retention stabilises from Month 5 (~{:.0f}%)".format(avg_ret[5]))
print("   - Long-term base: ~{:.0f}% of acquired subscribers".format(avg_ret[-1]))
print("")
print("2. COHORT TRENDS")
if len(m3) >= 2:
    z3 = np.polyfit(range(len(m3)), m3, 1)
    trend = 'improving' if z3[0] > 0 else 'declining'
    print("   - Month-3 retention is {} over 2024 (slope: {:+.2f}pp/cohort)".format(trend, z3[0]))
print("")
print("3. LTV")
print("   - Average LTV: GBP{:.2f}".format(ltv_df['LTV_GBP'].mean()))
print("   - Total estimated value (2024 cohorts): GBP{:,.0f}".format(
    (ltv_df['LTV_GBP'] * ltv_df['Cohort_Size']).sum()))
print("")
print("RECOMMENDATIONS")
print("  1. Prioritise Month-1 retention -- 28% of subscribers are lost here.")
print("     Action: improve onboarding email sequence and early content discovery.")
print("  2. Set Month-3 retention as a leading KPI for product health.")
print("  3. Build an early-warning churn model using Month-1 engagement signals.")
print("  4. Investigate later cohorts' higher LTV for acquisition insight.")